# Datathon Passos Mágicos — Análise PEDE (2022, 2023, 2024)

**Objetivo:** responder às 11 perguntas de negócio do desafio POSTECH – Datathon Fase 5, usando a base `BASE_DE_DADOS_PEDE_2024_-_DATATHON.xlsx` (abas `PEDE2022`, `PEDE2023`, `PEDE2024`) e o dicionário de dados/documentos de apoio fornecidos.

Cada seção traz: código de análise → visualização (quando aplicável) → **resposta objetiva embasada nos números obtidos**.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (9, 4.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

PATH = 'BASE DE DADOS PEDE 2024 - DATATHON.xlsx'  # ajuste o caminho se necessário

## 1. Carregamento e harmonização dos dados

As três abas têm esquemas de coluna diferentes (nomes de fase, notas, presença de `IPP`, etc.). Construímos uma base única **longitudinal** (`full`), com uma linha por aluno/ano, padronizando nomes de colunas e extraindo o número da fase (`Fase_num`) a partir de rótulos como `"FASE 3"`, `"3A"` ou `"ALFA"`.

In [4]:
def fase_num(x):
    """Extrai o número da fase a partir de rótulos como 'FASE 3', '3A' ou 'ALFA' (=0)."""
    if pd.isna(x):
        return np.nan
    s = str(x).upper().strip()
    if s.startswith('ALFA'):
        return 0
    m = re.search(r'(\d+)', s)
    return int(m.group(1)) if m else np.nan


def load_year(sheet, ano):
    df = pd.read_excel(PATH, sheet_name=sheet)
    out = pd.DataFrame()
    out['RA'] = df['RA']
    out['Ano'] = ano
    out['Fase_num'] = df['Fase'].apply(fase_num)
    out['Turma'] = df.get('Turma')
    out['Genero'] = df['Gênero'].replace({'Menina': 'Feminino', 'Menino': 'Masculino'})
    out['Ano_ingresso'] = df['Ano ingresso']

    inde_col = {2022: 'INDE 22', 2023: 'INDE 2023', 2024: 'INDE 2024'}[ano]
    out['INDE'] = pd.to_numeric(df[inde_col], errors='coerce')

    pedra_col = 'Pedra 22' if ano == 2022 else f'Pedra {ano}'
    out['Pedra'] = df[pedra_col].replace({'Agata': 'Ágata'})
    out.loc[out['Pedra'] == 'INCLUIR', 'Pedra'] = np.nan

    out['IAA'] = df['IAA']
    out['IEG'] = df['IEG']
    out['IPS'] = df['IPS']
    out['IPP'] = df['IPP'] if 'IPP' in df.columns else np.nan   # IPP só existe em 2023/2024
    out['IDA'] = df['IDA']
    out['IAN'] = df['IAN']
    out['IPV'] = df['IPV']
    out['Defasagem'] = df['Defas'] if 'Defas' in df.columns else df['Defasagem']
    out['Atingiu_PV'] = df['Atingiu PV'] if 'Atingiu PV' in df.columns else np.nan  # só 2022
    out['Indicado_bolsa'] = df['Indicado'] if 'Indicado' in df.columns else np.nan  # só 2022

    mat_col = 'Matem' if 'Matem' in df.columns else 'Mat'
    por_col = 'Portug' if 'Portug' in df.columns else 'Por'
    out['Nota_Mat'] = df[mat_col]
    out['Nota_Por'] = df[por_col]
    out['Instituicao'] = df['Instituição de ensino']
    return out


d22 = load_year('PEDE2022', 2022)
d23 = load_year('PEDE2023', 2023)
d24 = load_year('PEDE2024', 2024)
full = pd.concat([d22, d23, d24], ignore_index=True)

def severidade(d):
    if d >= 0:
        return 'Adequado (>=0)'
    if d == -1:
        return 'Moderado (-1 ano)'
    return 'Severo (<=-2 anos)'

full['Sev_defasagem'] = full['Defasagem'].apply(severidade)

print(f"Total de registros: {len(full)}  |  Alunos únicos: {full['RA'].nunique()}")
full.groupby('Ano').size().rename('nº alunos').to_frame()

**Limitações importantes dos dados** (impactam o que é possível responder por ano):
- `IPP` só existe nas abas 2023 e 2024 (ausente em 2022) → a pergunta 6 usa apenas 2023/2024.
- `Atingiu PV` (booleano) e `Indicado_bolsa` só existem em 2022 → nos demais anos usamos o **indicador contínuo `IPV`** como proxy do ponto de virada.
- Nem todo aluno aparece nos três anos (entradas/saídas do programa) — para as análises longitudinais (evolução do mesmo aluno) usamos apenas o subconjunto de alunos presentes em anos consecutivos, identificados pelo `RA`.

In [ ]:
def pares_longitudinais(full, ano_n, ano_n1, cols):
    """Casa o mesmo aluno (RA) entre dois anos consecutivos e retorna um DataFrame wide."""
    d1 = full[full['Ano'] == ano_n].set_index('RA')
    d2 = full[full['Ano'] == ano_n1].set_index('RA')
    comuns = d1.index.intersection(d2.index)
    out = pd.DataFrame(index=comuns)
    for c in cols:
        out[f'{c}_N'] = d1.loc[comuns, c]
        out[f'{c}_N1'] = d2.loc[comuns, c]
    return out

## Pergunta 1 — Adequação de nível (IAN): perfil de defasagem e evolução

In [ ]:
tab_defas = pd.crosstab(full['Ano'], full['Sev_defasagem'], normalize='index').mul(100).round(1)
tab_defas = tab_defas[['Adequado (>=0)', 'Moderado (-1 ano)', 'Severo (<=-2 anos)']]
ian_medio = full.groupby('Ano')['IAN'].mean().round(2)

print("% de alunos por severidade de defasagem:")
display(tab_defas)
print("\nIAN médio por ano:")
display(ian_medio)

tab_defas.plot(kind='bar', stacked=True, colormap='RdYlGn_r')
plt.title('Distribuição da defasagem de nível (Fase efetiva vs. ideal) por ano')
plt.ylabel('% de alunos'); plt.xlabel('Ano'); plt.xticks(rotation=0)
plt.legend(title='Severidade', bbox_to_anchor=(1.02,1), loc='upper left')
plt.tight_layout(); plt.show()

**Resposta:** o quadro de defasagem melhora consistentemente: alunos "adequados" (D≥0) saltam de **30,1% (2022) → 45,6% (2023) → 53,8% (2024)**, enquanto a defasagem severa (2+ anos) cai de **22,2% → 14,2% → 8,0%**. O IAN médio acompanha essa tendência (**6,42 → 7,24 → 7,68**). A defasagem moderada (1 ano) permanece o grupo mais estável (≈38–48%), sugerindo que a "última milha" para recolocar o aluno na fase ideal é o principal gargalo remanescente.

## Pergunta 2 — Desempenho acadêmico (IDA): melhora, estagna ou cai?

In [ ]:
ida_ano = full.groupby('Ano')['IDA'].mean().round(2)
ida_fase = full.pivot_table(index='Fase_num', columns='Ano', values='IDA', aggfunc='mean').round(2)

print("IDA médio geral por ano:"); display(ida_ano)
print("\nIDA médio por fase e ano:"); display(ida_fase)

ida_fase.plot(marker='o')
plt.title('IDA médio por fase, ao longo dos anos')
plt.xlabel('Fase (0=ALFA)'); plt.ylabel('IDA médio')
plt.legend(title='Ano'); plt.tight_layout(); plt.show()

**Resposta:** no agregado, o IDA médio oscila sem tendência clara: **6,09 (2022) → 6,66 (2023) → 6,35 (2024)** — uma leve melhora seguida de recuo, portanto **estagnado/instável**, não uma trajetória de melhora sustentada. Ao abrir por fase, o padrão é heterogêneo: fases iniciais (0–1, ALFA/Fase 1) mantêm IDA relativamente estável e mais alto (~6,5–7,4), enquanto fases intermediárias (2–4, tipicamente 5º–8º ano) ficam consistentemente mais baixas (~5,1–6,7), indicando que a perda de desempenho acadêmico se concentra na transição para os anos finais do fundamental.

## Pergunta 3 — Engajamento (IEG) x Desempenho (IDA) e Ponto de Virada (IPV)

In [ ]:
corr_ieg = full.groupby('Ano').apply(lambda d: pd.Series({
    'corr(IEG, IDA)': d['IEG'].corr(d['IDA']),
    'corr(IEG, IPV)': d['IEG'].corr(d['IPV'])
})).round(3)
display(corr_ieg)

fig, ax = plt.subplots(1, 2, figsize=(11,4.5))
ax[0].scatter(full['IEG'], full['IDA'], s=6, alpha=0.3)
ax[0].set_xlabel('IEG'); ax[0].set_ylabel('IDA'); ax[0].set_title('IEG x IDA')
ax[1].scatter(full['IEG'], full['IPV'], s=6, alpha=0.3, color='darkorange')
ax[1].set_xlabel('IEG'); ax[1].set_ylabel('IPV'); ax[1].set_title('IEG x IPV')
plt.tight_layout(); plt.show()

**Resposta:** sim, há relação direta e consistente nos três anos. A correlação de Pearson entre IEG e IDA varia de **0,46 a 0,56**, e entre IEG e IPV de **0,45 a 0,59** — força **moderada a moderada-forte**, estável ao longo do tempo. Engajamento não é apenas um indicador de "comportamento": ele se traduz em desempenho acadêmico e em progresso no ponto de virada, reforçando que ações de estímulo ao engajamento (presença, entrega de tarefas, atividades) têm efeito prático mensurável nos outros dois indicadores.

## Pergunta 4 — Autoavaliação (IAA) é coerente com desempenho (IDA) e engajamento (IEG)?

In [ ]:
corr_iaa = full.groupby('Ano').apply(lambda d: pd.Series({
    'corr(IAA, IDA)': d['IAA'].corr(d['IDA']),
    'corr(IAA, IEG)': d['IAA'].corr(d['IEG'])
})).round(3)
display(corr_iaa)

full['gap_confianca'] = full['IAA'] - full['IDA']
gap_pedra = full.groupby('Pedra')['gap_confianca'].mean().reindex(['Quartzo','Ágata','Ametista','Topázio']).round(2)
print("\nGap médio (IAA - IDA) por Pedra — quanto maior, mais o aluno se avalia acima do desempenho real:")
display(gap_pedra)

**Resposta:** a coerência é **baixa**. As correlações entre IAA e IDA (0,10–0,22) e entre IAA e IEG (0,17–0,32) são fracas em todos os anos — a autopercepção do aluno explica pouco da variação em desempenho real ou engajamento observado. O gap (IAA − IDA) é positivo em todas as pedras e **maior justamente nos alunos de menor desempenho** (Quartzo: +2,29; Ágata: +1,97) e menor nos de melhor desempenho (Topázio: +0,84) — um padrão clássico de excesso de confiança inversamente proporcional ao desempenho, que sinaliza a necessidade de calibrar a autoavaliação com feedback mais direto, especialmente nas fases de maior dificuldade.

## Pergunta 5 — Padrões psicossociais (IPS) antecedem quedas de desempenho/engajamento?

In [ ]:
cols = ['IPS','IDA','IEG']
pares = pd.concat([
    pares_longitudinais(full, 2022, 2023, cols),
    pares_longitudinais(full, 2023, 2024, cols)
])
pares['delta_IDA'] = pares['IDA_N1'] - pares['IDA_N']
pares['delta_IEG'] = pares['IEG_N1'] - pares['IEG_N']

print("Correlação IPS(ano N) com variação de IDA/IEG no ano seguinte:")
print('corr(IPS_N, delta_IDA) =', round(pares['IPS_N'].corr(pares['delta_IDA']), 3))
print('corr(IPS_N, delta_IEG) =', round(pares['IPS_N'].corr(pares['delta_IEG']), 3))

pares['IPS_baixo (1º quartil)'] = pares['IPS_N'] < pares['IPS_N'].quantile(0.25)
resumo = pares.groupby('IPS_baixo (1º quartil)')[['delta_IDA','delta_IEG']].mean().round(3)
print("\nVariação média no ano seguinte, por grupo de IPS inicial:")
display(resumo)

**Resposta:** sim — IPS baixo antecede piora relativa. Alunos no **1º quartil de IPS** no ano N apresentam, no ano seguinte, queda muito mais acentuada de IDA (**−0,63** vs. **−0,08** nos demais) e de IEG (**−0,82** vs. **−0,29**). As correlações diretas (IPS × Δ desempenho) são fracas isoladamente (0,12–0,16), mas o corte por quartil revela um efeito de limiar: **IPS baixo é um sinal de alerta precoce (early warning)** para queda de desempenho e engajamento no ano seguinte, útil como gatilho de intervenção psicossocial preventiva.

## Pergunta 6 — Avaliações psicopedagógicas (IPP) confirmam a defasagem do IAN?

In [ ]:
sub = full[full['Ano'].isin([2023, 2024])].copy()  # IPP só existe nesses anos
print('corr(IPP, IAN) =', round(sub['IPP'].corr(sub['IAN']), 3))
print('corr(IPP, Defasagem) =', round(sub['IPP'].corr(sub['Defasagem']), 3))

ipp_sev = sub.groupby('Sev_defasagem')['IPP'].mean().reindex(
    ['Adequado (>=0)','Moderado (-1 ano)','Severo (<=-2 anos)']).round(2)
print("\nIPP médio por severidade de defasagem (2023-2024):")
display(ipp_sev)

**Resposta:** há **confirmação direcional, mas fraca em magnitude**. IPP e IAN são positivamente correlacionados (0,12) e o IPP médio cai de forma monotônica com a severidade da defasagem: **7,68 (adequado) → 7,55 (moderado) → 7,11 (severo)**. Ou seja, quanto mais defasado o aluno está em relação à fase ideal, pior tende a ser sua avaliação psicopedagógica — as duas métricas apontam na mesma direção, mas a correlação fraca indica que **IPP e IAN capturam dimensões parcialmente distintas** (uma é estrutural/etária, a outra é qualitativa) e devem ser lidas em conjunto, não como substitutas uma da outra.

## Pergunta 7 — O que mais influencia o Ponto de Virada (IPV)?

In [ ]:
corr_ipv = full[['IAA','IEG','IPS','IPP','IAN','IDA','IPV']].corr()['IPV'].drop('IPV').sort_values(ascending=False).round(3)
print("Correlação de cada indicador com IPV:")
display(corr_ipv)

corr_ipv.plot(kind='barh', color='teal')
plt.title('Correlação de cada indicador com o IPV')
plt.xlabel('Correlação de Pearson'); plt.tight_layout(); plt.show()

**Resposta:** os indicadores **acadêmicos e de engajamento dominam** o Ponto de Virada: **IPP (0,61)**, **IEG (0,56)** e **IDA (0,56)** têm as correlações mais fortes com IPV, seguidos de longe por IAN (0,15). **IAA (0,06) e IPS (−0,05) praticamente não se relacionam** com IPV. Isso indica que o ponto de virada é impulsionado principalmente por comportamentos observáveis e mensuráveis pela equipe pedagógica (engajamento, aprendizagem, avaliação psicopedagógica) — não pela autopercepção do aluno nem, isoladamente, pelo aspecto psicossocial, que age mais como fator antecedente indireto (ver Pergunta 5) do que como componente direto do IPV.

## Pergunta 8 — Quais combinações de indicadores mais elevam o INDE?

In [ ]:
from sklearn.linear_model import LinearRegression

cols = ['IDA','IEG','IPS','IPP']
sub = full.dropna(subset=cols + ['INDE'])
X = (sub[cols] - sub[cols].mean()) / sub[cols].std()  # padronizado -> coeficientes comparáveis
y = sub['INDE']

lr = LinearRegression().fit(X, y)
pesos = pd.Series(lr.coef_, index=cols).sort_values(ascending=False).round(3)
print(f"R² do modelo linear (N={len(sub)}): {lr.score(X, y):.3f}\n")
print("Peso padronizado de cada indicador sobre o INDE:")
display(pesos)

pesos.plot(kind='bar', color='darkgreen')
plt.title('Peso relativo de cada indicador na explicação do INDE'); plt.ylabel('Coeficiente padronizado')
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

**Resposta:** um modelo linear com IDA, IEG, IPS e IPP explica **82,3% da variação do INDE** (R²=0,823, N=1.985). A combinação que mais eleva a nota global é, em ordem de peso: **IDA (0,45) > IEG (0,39) > IPS (0,23) ≈ IPP (0,21)**. Ou seja, desempenho acadêmico e engajamento têm impacto quase duas vezes maior que os componentes psicossocial/psicopedagógico — mas os quatro juntos, e não um isoladamente, é que sustentam um INDE alto, confirmando o caráter **multidimensional** do índice por desenho.

## Pergunta 9 — Modelo preditivo de risco de defasagem

**Definição do alvo:** para cada aluno com registro em dois anos consecutivos, usamos os indicadores do ano **N** para prever se ele estará **defasado (Defasagem ≤ −1) no ano N+1**. Isso simula o uso real do modelo: antecipar o risco *antes* de ele se concretizar.

### 9.1 Feature engineering

In [ ]:
cols_feat = ['IAA','IEG','IPS','IDA','IAN','IPV','Defasagem','Fase_num']
pares9 = pd.concat([
    pares_longitudinais(full, 2022, 2023, cols_feat),
    pares_longitudinais(full, 2023, 2024, cols_feat)
])
pares9 = pares9.rename(columns={'Fase_num_N':'Fase_num'})  # fase de referência = ano N
pares9['risco_defasagem'] = (pares9['Defasagem_N1'] <= -1).astype(int)

features = ['IAA_N','IEG_N','IPS_N','IDA_N','IAN_N','IPV_N','Defasagem_N','Fase_num']
dataset = pares9[features + ['risco_defasagem']].dropna()

print(f"N amostras: {len(dataset)}")
print(f"Taxa de risco (classe positiva): {dataset['risco_defasagem'].mean():.1%}")
dataset.head()

### 9.2 Separação treino/teste

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = dataset[features]
y = dataset['risco_defasagem']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f"Treino: {len(X_train)} | Teste: {len(X_test)}")

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

### 9.3 Modelagem preditiva

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, roc_curve

modelos = {
    'Regressão Logística': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42)
}

resultados = {}
for nome, modelo in modelos.items():
    if nome == 'Regressão Logística':
        modelo.fit(X_train_s, y_train)
        proba = modelo.predict_proba(X_test_s)[:, 1]
        pred = modelo.predict(X_test_s)
    else:
        modelo.fit(X_train, y_train)
        proba = modelo.predict_proba(X_test)[:, 1]
        pred = modelo.predict(X_test)
    auc = roc_auc_score(y_test, proba)
    resultados[nome] = {'modelo': modelo, 'proba': proba, 'pred': pred, 'auc': auc}
    print(f"=== {nome} (AUC = {auc:.3f}) ===")
    print(classification_report(y_test, pred, target_names=['Sem risco','Em risco']))

### 9.4 Avaliação e importância das variáveis

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

for nome, r in resultados.items():
    fpr, tpr, _ = roc_curve(y_test, r['proba'])
    ax[0].plot(fpr, tpr, label=f"{nome} (AUC={r['auc']:.3f})")
ax[0].plot([0,1],[0,1],'k--', alpha=0.4)
ax[0].set_xlabel('Falso positivo'); ax[0].set_ylabel('Verdadeiro positivo')
ax[0].set_title('Curva ROC'); ax[0].legend()

rf = resultados['Random Forest']['modelo']
importancias = pd.Series(rf.feature_importances_, index=features).sort_values()
importancias.plot(kind='barh', ax=ax[1], color='indianred')
ax[1].set_title('Importância das variáveis (Random Forest)')

plt.tight_layout(); plt.show()

**Resposta:** o modelo (Random Forest) atinge **AUC = 0,87** e **acurácia = 80%** no conjunto de teste (Regressão Logística: AUC=0,85, acurácia=75%), com precisão e recall balanceados (~0,79–0,81) para a classe "em risco" — desempenho sólido dado que a base de alunos que entram/saem do programa limita o tamanho da amostra pareada (N≈1.290). As variáveis mais relevantes para prever o risco de defasagem no ano seguinte são, em ordem: **Fase atual, IPV, Defasagem já existente no ano N, IDA e IEG** — ou seja, o principal preditor de defasagem futura é a combinação entre o estágio em que o aluno já está e sua trajetória recente de progresso/engajamento, mais do que autoavaliação (IAA) ou aspecto psicossocial (IPS) isoladamente. Esse modelo pode ser exportado (`pickle`/`joblib`) para alimentar a aplicação Streamlit de apoio à decisão da equipe pedagógica.

## Pergunta 10 — Efetividade do programa por fase (Pedra)

In [ ]:
order = ['Quartzo','Ágata','Ametista','Topázio']
inde_pedra = full.pivot_table(index='Pedra', columns='Ano', values='INDE', aggfunc='mean').reindex(order).round(2)
dist_pedra = pd.crosstab(full['Pedra'], full['Ano'], normalize='columns').reindex(order).mul(100).round(1)

print("INDE médio por Pedra e ano:"); display(inde_pedra)
print("\n% de alunos por Pedra e ano:"); display(dist_pedra)

# trajetória dos alunos presentes nos 3 anos
d22 = full[full['Ano']==2022].set_index('RA')
d23 = full[full['Ano']==2023].set_index('RA')
d24 = full[full['Ano']==2024].set_index('RA')
comuns = d22.index.intersection(d23.index).intersection(d24.index)
traj = pd.DataFrame({
    'INDE22': d22.loc[comuns,'INDE'], 'INDE23': d23.loc[comuns,'INDE'], 'INDE24': d24.loc[comuns,'INDE']
}).dropna()
print(f"\nAlunos com trajetória completa 2022-2024: {len(traj)}")
print("INDE médio na trajetória:", traj.mean().round(2).to_dict())
print(f"% que melhorou INDE de 2022 para 2024: {(traj['INDE24']>traj['INDE22']).mean():.1%}")

dist_pedra.T.plot(kind='bar', stacked=True, color=['#b0b0b0','#8fbc8f','#9370db','#ffd700'])
plt.title('Composição de Pedras por ano'); plt.ylabel('% de alunos'); plt.xticks(rotation=0)
plt.legend(title='Pedra', bbox_to_anchor=(1.02,1), loc='upper left'); plt.tight_layout(); plt.show()

**Resposta:** o INDE médio **dentro de cada Pedra é estável ao longo dos anos** (ex.: Topázio 8,37→8,44→8,47; Ametista ~7,5 nos três anos) — cada faixa mantém sua identidade de desempenho, o que valida a consistência da régua de classificação. O sinal mais forte de efetividade está na **migração de composição**: Topázio cresce de **15,1% → 24,9% → 29,9%** dos alunos, enquanto Quartzo cai de **15,3% → 7,7% → 10,6%**. Entre os 434 alunos com trajetória completa nos 3 anos, **55,3% aumentaram seu INDE** de 2022 para 2024 (INDE médio quase estável: 7,38→7,35→7,38, refletindo que grande parte da base já entra em níveis elevados). Em conjunto, os dados **confirmam impacto real do programa**, mais visível no deslocamento de alunos para faixas superiores do que numa alta homogênea do INDE médio geral.

## Pergunta 11 — Insights adicionais

In [ ]:
print("--- INDE médio por gênero e ano")
display(full.pivot_table(index='Genero', columns='Ano', values='INDE', aggfunc='mean').round(2))

print("\n--- Retenção de alunos entre anos consecutivos")
ra22, ra23, ra24 = (set(full[full['Ano']==a]['RA']) for a in (2022,2023,2024))
print(f"Retenção 2022->2023: {len(ra22 & ra23)/len(ra22):.1%}  |  Retenção 2023->2024: {len(ra23 & ra24)/len(ra23):.1%}")
print(f"Alunos novos em 2023: {len(ra23 - ra22)} de {len(ra23)} ({len(ra23-ra22)/len(ra23):.1%})")
print(f"Alunos novos em 2024: {len(ra24 - ra23)} de {len(ra24)} ({len(ra24-ra23)/len(ra24):.1%})")

print("\n--- INDE médio: indicados a bolsa vs. não indicados (2022)")
display(full[full['Ano']==2022].groupby('Indicado_bolsa')['INDE'].mean().round(2))

**Resposta:** três achados adicionais relevantes:

1. **Gênero:** meninas apresentam INDE ligeiramente superior aos meninos nos três anos (7,09 vs. 6,97 em 2022; 7,48 vs. 7,30 em 2024), uma diferença pequena mas consistente — não é uma alavanca prioritária, mas vale monitorar.
2. **Rotatividade da base:** cerca de **30–40% dos alunos são novos a cada ano** (414/1.014 em 2023; 391/1.156 em 2024) e a retenção ano a ano gira em torno de **70–75%**. Isso reduz o tamanho da coorte longitudinal disponível para medir evolução individual e deve ser considerado na leitura de qualquer indicador "de programa" — parte da mudança nos indicadores agregados reflete composição da turma, não apenas evolução dos mesmos alunos.
3. **Bolsa como sinal de qualidade, não de risco:** alunos indicados a bolsa em 2022 têm INDE médio mais alto (7,41) que os não indicados (6,97) — a indicação parece reconhecer desempenho já consolidado. Sugestão: cruzar a indicação de bolsa com o modelo de risco (Pergunta 9) para verificar se bons candidatos a bolsa em risco de defasagem estão sendo identificados a tempo.

## Conclusão e recomendações

- **O programa está funcionando na direção certa**: defasagem severa caiu de 22% para 8% dos alunos entre 2022 e 2024, e a proporção de alunos em Topázio quase dobrou.
- **Engajamento (IEG) é a alavanca mais acionável**: correlaciona-se fortemente com desempenho (IDA) e ponto de virada (IPV), e é mensurável e trabalhável no dia a dia da Associação.
- **IPS baixo é um alerta precoce confiável** de queda de desempenho no ano seguinte — recomenda-se transformar o 1º quartil de IPS em gatilho automático de acompanhamento psicossocial.
- **A autoavaliação (IAA) é pouco confiável isoladamente**, especialmente entre os alunos de menor desempenho (maior excesso de confiança) — deve ser complementada, não usada como proxy de desempenho real.
- **O modelo de risco de defasagem (AUC 0,87)** está pronto para operacionalização via aplicação Streamlit, usando como entrada os indicadores do ano corrente (IPV, IDA, IEG, Defasagem atual, Fase) para priorizar alunos antes da próxima avaliação.
- **Atenção à rotatividade da base** (~30% de novos alunos/ano): recomenda-se acompanhar coortes por tempo de permanência, não só por ano-calendário, para isolar o efeito real do programa da troca de composição.